# Literature review

`onceover` is a synthesis: nearly every pass implements, adapts, or
deliberately declines a known result. This appendix situates the
profiler in that literature — what it borrowed, from whom, and what it
chose to avoid. It is a reading list with verdicts, not a survey of the
fields.

## Data profiling and dependency discovery

The field's map is the survey by **Abedjan, Golab & Naumann**, *Profiling
Relational Data* (VLDB Journal, 2015), which tiers profiling into
single-column statistics, multi-column metadata (correlations, unique
column combinations, functional dependencies, inclusion dependencies),
and conditional properties. Mainstream tools stop at the first tier;
onceover's category set is essentially tiers one and two plus a bounded
selection of tier three.

- **Functional dependencies.** The discovery lineage runs through TANE
  (Huhtala et al., *Computer Journal*, 1999) — level-wise lattice search
  over partitions, with subset pruning — through the hybrid
  hash/lattice generation of HyFD (Papenbrock & Naumann, EDBT 2016)
  and DFD (Abedjan et al., CIKM 2014). onceover's pass is the hybrid in miniature: a hash-based screen
  proposes, a level-wise growth with subset pruning explores, and
  canonical value bytes verify — the same "propose cheaply, verify
  exactly" division of labor, at bounded depth.
- **Approximate FDs.** The g1 error (violations over tuples) is from
  Kivinen & Mannila (*Theoretical Computer Science*, 1995). That
  several AFD metrics disagree, and that g1 is the operational choice,
  is argued by Parciak et al. (ICDE 2024). Approximate *unique column
  combinations* — the dirty-key statement — follow Kruse & Naumann
  (PVLDB 2018).
- **Order dependencies.** The sortedness pass is the positive-claim
  fragment of the order-dependency literature (Langer & Naumann, VLDB
  Journal 2016): "increasing with row order", nothing stronger.
- **Arithmetic identities.** BHUNT (Brown & Haas, VLDB 2003) probes
  algebraic constraints by pair sampling; BACON (Langley, IJCAI 1977)
  discovered empirical laws by invariance detection. onceover's affine
  probe — fit on a sample, verify residuals on all rows — is the
  pairwise ancestor of both ideas.
- **Soft dependencies.** CORDS (Ilyas et al., SIGMOD 2004) used
  correlation to propose soft FDs cheaply; onceover's dependence screen
  plays the same proposal role for the identity probe and the regime
  pass.

**Desbordante** (AGPL-3.0) is the most complete dependency miner
available and is used as a reference point for behavior — its license
makes it unusable inside this MIT codebase, which is precisely the kind
of constraint that shaped the in-house implementations.

## Rule induction, subgroups, and rules

- **PRISM** (Cendrowska, *IJMMS* 1987) induces modular covering rules —
  one class at a time, specializing by the most informative condition.
  onceover implements it over bitset masks, with the near-majority guard
  and the shared evaluator layered on top — the adaptation, not the
  original, is what the report documents.
- **Decision tables** (Kohavi, ECML 1995) showed that tiny lookup
  subsets with cell-majority prediction rival far larger models. The
  pass here is the same recipe: greedy subset search, cross-validated
  cell-majority, a majority-class baseline.
- **CN2** (Clark & Niblett, 1989) and **CN2-SD** (Lavrač et al., JMLR
  2004) frame rule induction as beam search with a quality function —
  the vocabulary (coverage, precision, significance) that the rule
  evaluator's counts share. Subgroup discovery proper — WRAcc,
  diversity selection (van Leeuwen & Knobbe, DMKD 2012), self-sufficient
  itemsets (Webb, TKDD 2010) — is **not implemented**; the association
  miner's support/confidence/lift filters are the bounded fragment of
  that literature, and widening it is future work.
- **Holdout validation for rules** follows the practice argued by
  Megiddo & Srikant (KDD 1998): rules selected on data they were not
  fitted on. The association miner's train/holdout split is that
  pattern. The shared evaluator's diagnostic partition is disclosed, but
  discovery may already have used those rows.
- **GritBot** (RuleQuest) popularized conditional anomalies — the
  surprising value *given* a context. The behavior influenced the
  taxonomy (findings as auditable statements with their defining
  conditions) but no conditional-anomaly pass is implemented; the GPL
  source was never a reading option.

## Piecewise relationships and changepoints

- **Chow** (1960) is the one-break test; the residual-shift pass is a
  centered cumulative-sum scan variant, evaluated on a holdout, with no
  p-value — deliberately narrower than the inferential lineage.
- **M5** (Quinlan) and **M5'** (Wang & Witten) are the model-tree
  concept: linear models in leaves, pruned and smoothed. onceover's
  regime pass is a bounded M5-style tree — standardization on training
  rows, QR-fitted leaves, corrected-error pruning, ancestor smoothing —
  documented as a variant, with no claim of Weka parity.
- **Muggeo** (*Statistics in Medicine*, 2003) formalized segmented
  regression; the supF/strucchange line (Zeileis et al.) and MOB
  (Zeileis, Hothorn & Hornik, JCGS 2008) extend to inferential and
  parameter-instability testing. Multiple breaks (PELT — Killick,
  Fearnhead & Eckley, JASA 2012; the review by Truong et al., *Signal
  Processing*, 2020) remain future work ([](directions.ipynb)).

## Dependence measures

**Chatterjee's ξ** (*JASA*, 2021) is the screen's most
load-bearing import: an O(n log n), asymmetric, rank-based coefficient
of "Y is a function of X" that detects arbitrary functional shapes.
The implementation follows the tie-corrected estimator, with seeded
random tie-breaking — the fixed-order variant is biased under row
position correlation, a failure the regression tests pin. Pearson and
Spearman need no citation but earn their keep as the cheap signals.
MIC (Reshef et al., *Science*, 2011) was considered and declined: its
reference implementation is dormant and GPL, and the equitability
critique (Kinney & Atwal, 2014) argues it does not beat ξ for the
profiler's purpose. Discretization for the encoded matrix is quantile
binning with fixed bin counts, the histogram-boosting convention
(Ke et al., NeurIPS 2017), rather than supervised MDLPC (Fayyad &
Irani, 1993) — unsupervised profiling has no target to supervise the
cuts.

## The tools landscape

The tool survey that motivated the package: **ydata-profiling**,
**sweetviz**, **dtale**, **skimpy** (single-column HTML-first profiling,
pandas-bound); **whylogs** (sketch-based, single-column); **DataProfiler**
(Capital One; schema/stats/PII); **TFDV** (Beam-oriented statistics and
schema validation); **Great Expectations** (a validation framework — it
checks what you already know to ask); **skrub**'s TableReport (the
closest prior art: Polars input, JSON, agent-facing markdown — and
column statistics is where it stops). None of them ship tier-two
dependency discovery at speed, and none treat coverage — what was *not*
examined — as a product feature. `polars` itself ships `describe()`,
explicitly documented as unstable for programmatic use.

## What this review is for

Two practical uses. First, provenance: every finding's evidence record
names its algorithm, and this appendix is where those names stop being
strings and become citations. Second, scoping: the
[directions](directions.ipynb) chapter lists what the literature offers
that the profiler has *not* adopted — the current implementation boundary.